# Theory figures and input spectrum

Filter-bank figures F5_1 to F5_3 and the SHD input spectrum F4_13.

In [ ]:
import sys
sys.path.append("../code")
import json
import numpy as np
import matplotlib.pyplot as plt
import common as C, stats as st, plots as P
P.apply()

## filter bank

In [ ]:
t = np.linspace(0, 300, 900)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, taus, title, col in [
    (axes[0], [5.0] * 6, "homogeneous: one window, repeated", P.HOM),
    (axes[1], np.geomspace(3.3, 199.5, 6), "heterogeneous: a spread of windows", P.HET),
]:
    for i, tt in enumerate(taus):
        k = np.exp(-t / tt) / tt
        ax.plot(t, k / k.max(), color=col, alpha=0.35 + 0.11 * i, lw=1.8)
    ax.set_xlabel("time since input (ms)")
    ax.set_title(title)
axes[0].set_ylabel(r"normalised kernel $k_\tau(t)$")
fig.suptitle("The leaky membrane as a filter bank", fontweight="bold", y=1.02)
P.save(fig, C.FIGDIR / "F5_1_filter_bank.png")

## sech kernel

In [ ]:
d = np.linspace(-8, 8, 500)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(d, 1 / np.cosh(d / 2), color=P.ACCENT, lw=2.2)
ax.axvline(0, color=P.CHANCE, ls=":", lw=1.1)
# ratios reported in the text
for r, lab in [(np.log(2), "2x"), (np.log(10), "10x"), (np.log(35), "35x")]:
    ax.plot([r], [1 / np.cosh(r / 2)], "o", color=P.HET, ms=6)
    ax.annotate(f"{lab} apart\nC = {1 / np.cosh(r / 2):.2f}", (r, 1 / np.cosh(r / 2)),
                xytext=(8, 8), textcoords="offset points", fontsize=8, color=P.ACCENT)
ax.set_xlabel(r"log-tau distance  $\ln\tau_i - \ln\tau_j$")
ax.set_ylabel(r"filter correlation  $C_{ij}$")
ax.set_title(r"Filter correlation $C_{ij}=\mathrm{sech}((u_i-u_j)/2)$")
P.save(fig, C.FIGDIR / "F5_2_sech_kernel.png")

## pr vs spread

In [ ]:
N = 64


def corr_matrix(tau):
    u = np.log(np.asarray(tau, dtype=np.float64))
    return 1.0 / np.cosh((u[:, None] - u[None, :]) / 2.0)


def pr_bank(tau):
    M = corr_matrix(tau)
    n = M.shape[0]
    return (n ** 2) / (n + 2 * np.triu(M ** 2, 1).sum())


tau_ref = np.array(C.load_record(202, "het")["tau_all_ms"])
u = np.log(tau_ref)
ubar = u.mean()
svals = np.linspace(0, 6, 160)
prs = [pr_bank(np.exp(ubar + s * (u - ubar))) for s in svals]

fig, ax = plt.subplots(figsize=(6.8, 4.4))
ax.plot(svals, prs, color=P.ACCENT, lw=2.2)
ax.axhline(N, color=P.CHANCE, ls="--", lw=1.2, label=f"N = {N} (fully decorrelated bank)")
ax.axhline(1, color=P.CHANCE, ls=":", lw=1.2, label="PR = 1 (homogeneous, s = 0)")
ax.plot([1.0], [pr_bank(tau_ref)], "o", color=P.HET, ms=9, zorder=5,
        label=f"seed 202's learned spread (s = 1): PR = {pr_bank(tau_ref):.1f}")
ax.set_xlabel("dilation of the log-tau configuration about its mean, $s$")
ax.set_ylabel("effective dimension of the filter bank, PR")
ax.set_title("Participation ratio of the filter bank vs log-tau spread")
ax.legend(loc="lower right", fontsize=8)
P.save(fig, C.FIGDIR / "F5_3_pr_vs_spread.png")

## input spectrum

In [ ]:
DT = 1e-3
NB_STEPS = 1000
shd = C.SHD / "shd_test.h5"
if not shd.exists():
    # the psd itself is not cached; without the raw h5 only the shipped summary is available
    print("shd_test.h5 not found, figure skipped")
    print(json.dumps(json.load(open(C.RESULTS / "input_spectrum.json")), indent=1))
else:
    import h5py
    with h5py.File(shd, "r") as f:
        trial_times = [np.asarray(x) for x in f["spikes"]["times"]]
    # population input per trial: spike count per 1 ms step over the 700 channels
    psds = []
    for ti in trial_times:
        idx = np.round(ti / DT).astype(np.int64)
        idx = idx[(idx >= 0) & (idx < NB_STEPS)]
        sig = np.bincount(idx, minlength=NB_STEPS).astype(np.float64)
        sig -= sig.mean()                   # drop dc
        psds.append(np.abs(np.fft.rfft(sig * np.hanning(NB_STEPS))) ** 2)
    PSD = np.mean(psds, axis=0)
    freqs = np.fft.rfftfreq(NB_STEPS, DT)

    band = (freqs > 0) & (freqs <= 200)
    centroid = float((freqs[band] * PSD[band]).sum() / PSD[band].sum())
    tau_centroid = 1.0 / (2 * np.pi * centroid) * 1e3
    cum = np.cumsum(PSD[band]) / PSD[band].sum()
    f10 = float(freqs[band][np.searchsorted(cum, 0.10)])
    f90 = float(freqs[band][np.searchsorted(cum, 0.90)])
    print(f"spectral centroid = {centroid:.2f} Hz -> tau = 1/(2 pi f) = {tau_centroid:.1f} ms")
    print(f"10-90% power band = {f10:.1f} to {f90:.1f} Hz")

    het_tau = np.concatenate([np.array(C.load_record(s, "het")["tau_all_ms"]) for s in C.SEEDS])
    hom_tau = np.array([C.load_record(s, "hom")["tau_median_ms"] for s in C.SEEDS])
    het_f = 1.0 / (2 * np.pi * het_tau * 1e-3)
    hom_f = 1.0 / (2 * np.pi * hom_tau * 1e-3)

    fig, ax = plt.subplots(figsize=(8.4, 5))
    ax.semilogx(freqs[band], PSD[band] / PSD[band].max(), color=P.ACCENT, lw=1.6,
                label="SHD test input PSD (normalised)")
    ax.axvline(centroid, color=P.CHANCE, ls="--", lw=1.5,
               label=f"spectral centroid {centroid:.1f} Hz (tau = {tau_centroid:.0f} ms)")
    ax.axvspan(f10, f90, color=P.CHANCE, alpha=0.10, lw=0, label="10 to 90% of input power")

    ax2 = ax.twinx()
    ax2.hist(het_f, bins=np.logspace(np.log10(0.7), np.log10(60), 34), color=P.HET, alpha=0.55,
             label="het corner frequencies (per neuron)")
    for f_ in hom_f:
        ax2.axvline(f_, color=P.HOM, alpha=0.65, lw=1.3)
    ax2.axvline(np.nan, color=P.HOM, lw=1.3, label="hom corner frequency (one per seed)")
    ax2.set_ylabel("number of hidden units")
    ax2.grid(False)

    ax.set_xlim(0.7, 200)
    ax.set_xlabel("frequency (Hz, log scale)")
    ax.set_ylabel("normalised input power")
    ax.set_title("Input spectrum against the learned filter bank")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)
    P.save(fig, C.FIGDIR / "F4_13_input_spectrum.png")

    json.dump({"centroid_hz": centroid, "tau_centroid_ms": tau_centroid, "f10_hz": f10, "f90_hz": f90,
               "tau_at_f90_ms": 1 / (2 * np.pi * f90) * 1e3, "tau_at_f10_ms": 1 / (2 * np.pi * f10) * 1e3,
               "het_frac_in_band": float(((het_f >= f10) & (het_f <= f90)).mean()),
               "hom_frac_in_band": float(((hom_f >= f10) & (hom_f <= f90)).mean())},
              open(C.RESULTS / "input_spectrum.json", "w"), indent=1)